# Ooma Lead Validation Pilot

This notebook is the live first-lane control desk for the DTO Ooma intake workflow.

It ingests a raw CSV batch, validates and normalizes each record, routes accepted leads, and preserves a reviewable evidence bundle for every run.

In [ ]:
from pathlib import Path
from collections import Counter
import csv
import json

import pandas as pd

from dpo_system.src.ooma_intake_pack import (
    process_ooma_csv_file,
    write_json,
    write_rejections_csv,
)

ROOT = Path.cwd()
BATCH_DIR = ROOT / 'dpo_system' / 'config' / 'outbound_batches'
REPORT_DIR = ROOT / 'dpo_system' / 'evidence' / 'EXECUTED_EVIDENCE' / 'ooma'
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def export_ooma_artifacts(batch_path: Path, run_label: str = 'ooma_pilot_run') -> dict:
    result = process_ooma_csv_file(batch_path)

    approved_path = REPORT_DIR / f'{run_label}_approved.csv'
    rejected_path = REPORT_DIR / f'{run_label}_rejected.csv'
    json_path = REPORT_DIR / f'{run_label}.json'

    if result['accepted']:
        fieldnames = [
            'lead_id',
            'full_name',
            'phone',
            'lane',
            'status',
            'last_call_result',
            'next_action',
            'notes',
            'aux',
            'source',
        ]
        with approved_path.open('w', encoding='utf-8', newline='') as handle:
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(result['accepted'])

    if result['rejected']:
        write_rejections_csv(rejected_path, result['rejected'])

    write_json(json_path, result)

    return {
        'batch_path': str(batch_path),
        'result': result,
        'approved_csv': str(approved_path),
        'rejected_csv': str(rejected_path),
        'evidence_json': str(json_path),
    }

print('Ooma pilot initialized. Ready to run the active batch flow.')

In [ ]:
# Run the actual Ooma batch that is in the repo.
BATCH_CSV = BATCH_DIR / 'ooma_batch_live.csv'
run_result = export_ooma_artifacts(BATCH_CSV, run_label='ooma_pilot_run')
run_result['result']['summary']

In [ ]:
accepted_df = pd.DataFrame(run_result['result']['accepted'])
rejected_df = pd.DataFrame(run_result['result']['rejected'])

rejection_counts = Counter(item['rejection_reason'] for item in run_result['result']['rejected'])
lane_counts = Counter(item['lane'] for item in run_result['result']['accepted'])

{
    'accepted_count': len(accepted_df),
    'rejected_count': len(rejected_df),
    'rejection_counts': dict(rejection_counts),
    'lane_counts': dict(lane_counts),
    'sample_accepted': accepted_df.head(5).to_dict(orient='records'),
    'sample_rejected': rejected_df.head(5).to_dict(orient='records'),
}

In [ ]:
approved_csv = Path(run_result['approved_csv'])
rejected_csv = Path(run_result['rejected_csv'])
evidence_json = Path(run_result['evidence_json'])

{
    'approved_csv': approved_csv,
    'rejected_csv': rejected_csv,
    'evidence_json': evidence_json,
    'approved_exists': approved_csv.exists(),
    'rejected_exists': rejected_csv.exists(),
    'evidence_exists': evidence_json.exists(),
}

## Pilot run review

Use this section to confirm the batch is ready to move forward:

- Accepted count and rejected count are understood
- Rejection reasons are reviewed for systemic issues
- Founder and BD lane counts are checked
- Approved CSV and rejected CSV are exported
- Evidence JSON is saved for traceability and operator signoff

If the batch is acceptable, record the outcome and move the approved records into the next operational step.

# Ooma lead validation pilot

This notebook is the active control surface for the first production-ready Ooma validation lane.

Operational objective:
- load a raw Ooma CSV batch
- normalize and validate records
- classify accepted and rejected rows
- route accepted leads by lane
- let an operator review the batch
- export approved and rejected artifact sets
- preserve review evidence for auditability

## Ooma pilot: live workflow

This notebook should be used as the operational pilot surface rather than a generic QA scratchpad.

The workflow is:
1. load raw CSV
2. validate required columns and row data
3. normalize names and phone numbers
4. assign lane using urgency keywords
5. separate accepted vs rejected records
6. review the approved batch with an operator
7. export a sanitized approved dataset and a rejection log
8. write the evidence bundle for traceability

This is the first proof that DTO can convert raw contact data into a governed, reviewable lead lane.